# LifeOS / Aven — Phase 2 Industry-Grade Wake Word Training (`hey_aven.onnx`)

**Runtime requirement:** Select `Runtime -> Change runtime type -> T4 GPU`.

1. Installs `openWakeWord`, `piper-sample-generator`, MIT RIRs, and precomputed `ACAV100M` negative features.
2. Lets you optionally upload a ZIP of your `dataset_real/` recordings recorded with `pnpm wake:record`.
3. Synthesizes 15,000+ multi-speaker variations + hard negatives (`even`, `when`, `been`, `open`, `fine`, `he won`, `haven`, `seven`, `Kevin`).
4. Trains, benchmarks blind accuracy, and downloads `hey_aven.onnx` + ONNX runtime assets.

In [ ]:
#@title 1. Install dependencies & clone openWakeWord / Piper generator
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!pip install -q audiomentations torch-audiomentations pronouncing datasets scipy tqdm pyyaml soundfile librosa onnx onnxruntime huggingface_hub piper-tts webrtcvad
!git clone https://github.com/rhasspy/piper-sample-generator
!mkdir -p piper-sample-generator/models models work
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
!git lfs install
!git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses ./work/mit_rirs

In [ ]:
#@title 2. Download base ONNX models & ACAV100M negative feature bank (~2,000 hrs)
import os, urllib.request
from huggingface_hub import hf_hub_download

base_urls = {
    'models/melspectrogram.onnx': 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx',
    'models/embedding_model.onnx': 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx',
    'models/silero_vad.onnx': 'https://github.com/snakers4/silero-vad/raw/master/src/silero_vad/data/silero_vad.onnx',
    'models/wespeaker_en_voxceleb_resnet34.onnx': 'https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-recongition-models/wespeaker_en_voxceleb_resnet34.onnx',
}
for dest, url in base_urls.items():
    if not os.path.exists(dest):
        urllib.request.urlretrieve(url, dest)

for fname in ['openwakeword_features_ACAV100M_2000_hrs_16bit.npy', 'validation_set_features.npy']:
    if not os.path.exists(f'./work/{fname}'):
        hf_hub_download(repo_id='davidscripka/openwakeword_features', filename=fname, repo_type='dataset', local_dir='./work')

In [ ]:
#@title 3. Upload zipped dataset_real.zip & config/train_pipeline.py from your PC
from google.colab import files
import zipfile

print('Upload dataset_real.zip (created by zipping tools/aven-wake-detector/training/dataset_real), config_hey_aven.yaml, and train_pipeline.py')
uploaded = files.upload()
if 'dataset_real.zip' in uploaded:
    with zipfile.ZipFile('dataset_real.zip', 'r') as zf:
        zf.extractall('.')
    print('Extracted dataset_real/')

In [ ]:
#@title 4. Train hey_aven.onnx & run blind adversarial oracle benchmark
!python3 train_pipeline.py

In [ ]:
#@title 5. Download trained model bundle back to PC
import os, zipfile
from google.colab import files

candidates = {
    'hey_aven.onnx': ['work/hey_aven_model/hey_aven.onnx', '../models/hey_aven.onnx', 'models/hey_aven.onnx'],
    'melspectrogram.onnx': ['models/melspectrogram.onnx'],
    'embedding_model.onnx': ['models/embedding_model.onnx'],
    'silero_vad.onnx': ['models/silero_vad.onnx'],
    'wespeaker_en_voxceleb_resnet34.onnx': ['models/wespeaker_en_voxceleb_resnet34.onnx'],
}
with zipfile.ZipFile('hey_aven_onnx_bundle.zip', 'w') as zf:
    for fn, paths in candidates.items():
        for p in paths:
            if os.path.exists(p):
                zf.write(p, arcname=fn)
                break
files.download('hey_aven_onnx_bundle.zip')